# Homework 1 — Agentic RAG
**LLM Zoomcamp 2026 | Cohort 2026**

Stack: gitsource · minsearch · Groq (llama-3.3-70b) · toyaikit

In [1]:
# ── Celda 1: Setup ───────────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=Path("/home/juangraciano/Documentos/llm-zoomcamp/my-homework/01-agentic-rag/.env"))

api_key = os.environ.get("GROQ_API_KEY", "")
print(f"API key cargada: {api_key[:10]}...")

# Groq es 100% compatible con el cliente OpenAI — solo cambia base_url
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
print("Cliente Groq listo")

API key cargada: gsk_UI1S5f...
Cliente Groq listo


In [2]:
# ── Celda 2: Q1 — Cargar documentos ─────────────────────────────────
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Q1 — Total de páginas: {len(documents)}")  # → 72

Q1 — Total de páginas: 72


In [3]:
# ── Celda 3: Q2 — Indexar y buscar ──────────────────────────────────
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)
print("Índice listo con", len(documents), "documentos")

query_q2 = "How does the agentic loop keep calling the model until it stops?"
results_q2 = index.search(query_q2, num_results=5)

print(f"\nQ2 — Primer resultado: {results_q2[0]['filename']}")
for i, r in enumerate(results_q2):
    print(f"  {i+1}. {r['filename']}")

Índice listo con 72 documentos

Q2 — Primer resultado: 01-agentic-rag/lessons/14-agentic-loop.md
  1. 01-agentic-rag/lessons/14-agentic-loop.md
  2. 01-agentic-rag/lessons/15-frameworks.md
  3. 01-agentic-rag/lessons/13-function-calling.md
  4. 01-agentic-rag/lessons/11-agents-intro.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md


In [4]:
# ── Celda 4: Q3 — RAG completo → contar tokens ──────────────────────

INSTRUCTIONS = """Your task is to answer questions from course participants
based on the provided context. If the answer is not found in the context,
respond with 'I don't know.'"""

def build_context(results):
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}")
        lines.append(doc['content'])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(query, results):
    return f"QUESTION: {query}\n\nCONTEXT:\n{build_context(results)}"

def rag(query, search_index, num_results=5):
    results = search_index.search(query, num_results=num_results)
    prompt  = build_prompt(query, results)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user",   "content": prompt}
        ]
    )
    return response.choices[0].message.content, response.usage.prompt_tokens

query_q3 = "How does the agentic loop keep calling the model until it stops?"
answer_q3, tokens_q3 = rag(query_q3, index)

print(f"Q3 — Tokens de entrada: {tokens_q3}")
print(f"\nRespuesta:\n{answer_q3[:300]}...")

Q3 — Tokens de entrada: 7207

Respuesta:
I'm ready to answer your questions based on the provided context. Go ahead and ask your question. If the answer is not found in the context, I will respond with 'I don't know.'...


In [5]:
# ── Celda 5: Q4 — Chunking ──────────────────────────────────────────
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4 — Total de chunks: {len(chunks)}")  # → 295

Q4 — Total de chunks: 295


In [6]:
# ── Celda 6: Q5 — RAG con chunks → comparar tokens ──────────────────
index_chunks = Index(text_fields=["content"], keyword_fields=["filename"])
index_chunks.fit(chunks)
print("Índice de chunks listo con", len(chunks), "chunks")

query_q5 = "How does the agentic loop keep calling the model until it stops?"
answer_q5, tokens_q5 = rag(query_q5, index_chunks)

print(f"\nQ3 — Tokens con páginas completas : {tokens_q3}")
print(f"Q5 — Tokens con chunks           : {tokens_q5}")
print(f"Reducción: {tokens_q3 / tokens_q5:.1f}x menos tokens con chunks")

Índice de chunks listo con 295 chunks

Q3 — Tokens con páginas completas : 7207
Q5 — Tokens con chunks           : 2334
Reducción: 3.1x menos tokens con chunks


In [8]:
# ── Celda 7: Q6 — Agentic loop desde cero (compatible con Groq) ─────
import json

search_call_count = 0

def search(query: str) -> str:
    """Search the course knowledge base for relevant content."""
    global search_call_count
    search_call_count += 1
    results = index_chunks.search(query, num_results=3)
    return "\n---\n".join([f"File: {r['filename']}\n{r['content'][:400]}" for r in results])

# Schema de la herramienta — le dice al LLM qué herramientas tiene disponibles
tools_schema = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course knowledge base for relevant content.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    }
}]

AGENT_INSTRUCTIONS = """You're a course teaching assistant. Answer the student's 
question using the search tool. Make multiple searches with different keywords 
before answering."""

def run_agent(question):
    """Agentic loop: llama al LLM, ejecuta tools, repite hasta que pare."""
    messages = [
        {"role": "system", "content": AGENT_INSTRUCTIONS},
        {"role": "user",   "content": question}
    ]
    
    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        messages.append(msg)
        
        # Si no hay tool calls → el modelo terminó, devolvemos la respuesta
        if not msg.tool_calls:
            return msg.content
        
        # Si hay tool calls → ejecutamos cada una y agregamos el resultado
        for tool_call in msg.tool_calls:
            result = search(tool_call.function.arguments 
                           if isinstance(tool_call.function.arguments, str) 
                           else json.dumps(tool_call.function.arguments))
            # Parseamos el query del JSON que mandó el modelo
            args = json.loads(tool_call.function.arguments)
            result = search(args["query"])
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

# Ejecutamos
search_call_count = 0
answer_q6 = run_agent("How does the agentic loop work, and how is it different from plain RAG?")

print(f"Q6 — Veces que el agente llamó a search: {search_call_count}")
print(f"\nRespuesta:\n{answer_q6[:400]}...")


Q6 — Veces que el agente llamó a search: 6

Respuesta:
The agentic loop is a workflow that involves a continuous loop of interactions between a user and an AI model, where the model generates responses and the user provides feedback, which is then used to refine the model's responses. This loop allows the model to learn and adapt to the user's needs, enabling more effective and efficient communication.

In contrast, plain RAG (Retrieve, Augment, Gener...
